# Light readiness decision — stage 2

**Question.** Do the executed assembly, resolution, definition and checkpoint tests justify implementing light?

**Hypothesis.** A longer observation horizon may resolve apparent non-assembly, while high-speed numerical sensitivity can remain an independent blocker.

**Controls.** This notebook changes no biological or numerical parameter. It reads the saved results of notebooks 01–05 and reports their different horizons and cohorts separately. No light field or gradient information is introduced.

**Decision criteria.** Assembly and exact checkpoints are necessary but not sufficient. The timestep must satisfy the explicitly declared practical metric/rate criteria over speeds 1, 3 and 5 BL/s. Those tolerances are diagnostic conventions, not biological constants or a formal convergence proof. Run notebooks 01–05 first.


In [1]:
from pathlib import Path
import sys

MODEL_DIR = next(
    folder for parent in (Path.cwd(), *Path.cwd().parents)
    for folder in (parent, parent / "papers")
    if (folder / "shiner_model.py").is_file() and (folder / "handlings").is_dir()
)
if str(MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(MODEL_DIR))
HERE = MODEL_DIR / "handlings" / "trial_two"
STAGE_DIR = HERE

import json
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

sys.path.insert(0, str(MODEL_DIR))
DATA_DIR = STAGE_DIR / "data"
long_runs = pd.read_csv(DATA_DIR / "01_long_horizon_summary.csv")
refinement = pd.read_csv(DATA_DIR / "02_refinement_comparison.csv")
resolution = json.loads((DATA_DIR / "02_timestep_readiness.json").read_text())
stride = json.loads((DATA_DIR / "02_recording_stride_validation.json").read_text())
definition = json.loads((DATA_DIR / "03_assembly_definition.json").read_text())
checkpoints = pd.read_csv(DATA_DIR / "04_checkpoint_validation.csv")
noise = pd.read_csv(DATA_DIR / "05_noise_holdout_summary.csv")
noise_comparison = pd.read_csv(DATA_DIR / "05_noise_holdout_comparison.csv")
assert len(long_runs) == 30 and len(checkpoints) == 30 and len(noise) == 40
assert resolution["ScientificRuns"] == 45 and stride["AllChecksPassed"]
print("Evidence loaded from five executed notebooks. This notebook adds no dynamics or light.")


Evidence loaded from five executed notebooks. This notebook adds no dynamics or light.


## Assembly time and censoring

The 180-s cohort uses 30 explicit random seeds. AssemblyTime is retrospective onset of an episode later confirmed to persist for 5 s; checkpoints are saved at confirmation. An unsuccessful finite run would be right-censored, never described as structurally unable to assemble.


In [2]:
successful = long_runs.loc[long_runs.AssemblySuccess]
assembly_stats = pd.DataFrame([{
    "Runs": len(long_runs), "Assembled": len(successful),
    "Censored": int(long_runs.Censored.sum()),
    "Assembly onset median (s)": successful.AssemblyTime.median(),
    "Q1 (s)": successful.AssemblyTime.quantile(.25),
    "Q3 (s)": successful.AssemblyTime.quantile(.75),
    "Min (s)": successful.AssemblyTime.min(), "Max (s)": successful.AssemblyTime.max(),
    "Post-confirmation breakup runs": int(successful.PostAssemblyBreakup.sum()),
}])
display(assembly_stats)
display(long_runs.loc[long_runs.Seed.isin([0, 2]),
    ["Seed", "AssemblyTime", "AssemblyConfirmedTime", "Censored", "PostAssemblyBreakup", "FinalComponentSizes"]])
assert long_runs.AssemblySuccess.all() and not long_runs.Censored.any()
assert not long_runs.PostAssemblyBreakup.any()


,Runs,Assembled,Censored,Assembly onset median (s),Q1 (s),Q3 (s),Min (s),Max (s),Post-confirmation breakup runs
0,30,30,0,28.5,13.1875,55.125,0.0,86.75,0


,Seed,AssemblyTime,AssemblyConfirmedTime,Censored,PostAssemblyBreakup,FinalComponentSizes
0,0,86.750,91.750,False,False,20
2,2,73.875,78.875,False,False,20


## Numerical resolution and the observation schedule

The 45 refinement cases compare equal 31.25-s durations and a common 0.125-s observation grid. Ensemble changes and matched-seed absolute changes both matter. The original instantaneous-turn social model also changes decision frequency as dt changes, so this is a practical model-resolution test rather than formal ODE convergence.


In [3]:
display(refinement[["Speed", "Metric", "EnsembleRelativeChange", "PairedRelativeChange", "MetricPass"]])
display(pd.DataFrame(resolution["SpeedReports"]))
assert resolution["CandidateDt_s"] == 0.03125
assert resolution["ReferenceDt_s"] == 0.015625
print("Candidate accepted across all tested speeds:", resolution["CandidateAcceptedAcrossAllSpeeds"])
print("Identical physics with recording every four updates:", stride["AllChecksPassed"])


,Speed,Metric,EnsembleRelativeChange,PairedRelativeChange,MetricPass
0,1.0,Polarization,0.032333,0.032333,True
1,1.0,LargestGroupFraction,0.007063,0.007063,True
2,1.0,MeanNND,0.248544,0.248779,False
3,1.0,MeanDistanceToCentroid,0.001195,0.247861,False
4,3.0,Polarization,0.129235,0.179243,False
5,3.0,LargestGroupFraction,0.100000,0.122222,False
6,3.0,MeanNND,0.300446,0.300446,False
7,3.0,MeanDistanceToCentroid,0.473378,0.674876,False
8,5.0,Polarization,0.002549,0.053267,True
9,5.0,LargestGroupFraction,0.019992,0.019992,True


,Speed,MetricCriteriaPassed,CandidateMissedPassRate,FinestMissedPassRate,CandidateMissedPassCriterionPassed,CandidateAccepted,FailedMetrics
0,1.0,False,0.0000,0.0000,True,False,"[MeanNND, MeanDistanceToCentroid]"
1,3.0,False,0.0128,0.0128,False,False,"[Polarization, LargestGroupFraction, MeanNND, ..."
2,5.0,False,0.0896,0.0192,False,False,[MeanDistanceToCentroid]


Candidate accepted across all tested speeds: False
Identical physics with recording every four updates: True


## Checkpoints and held-out noise

The saved states come from the first qualifying persistent assembly, without selecting a later attractive frame. Source continuations validate more than two copies of the same reconstructed model. The noise comparison uses seeds 30–49, held out from the original ten and the 30-seed main cohort. It is exploratory and does not optimize noise strength.


In [4]:
verification_columns = ["StateRoundTripExact", "OriginalContinuationExact", "PairedControlsExact"]
assert checkpoints[verification_columns].all().all()
assert checkpoints.OriginalContinuationSteps.eq(16).all()
assert checkpoints.PairedControlUpdates.eq(64).all()
display(checkpoints[["Seed", "CheckpointTime_s", "dt", *verification_columns]])
display(noise_comparison)


,Seed,CheckpointTime_s,dt,StateRoundTripExact,OriginalContinuationExact,PairedControlsExact
0,0,91.750,0.125,True,True,True
1,1,5.000,0.125,True,True,True
2,2,78.875,0.125,True,True,True
3,3,19.250,0.125,True,True,True
4,4,52.000,0.125,True,True,True
5,5,17.500,0.125,True,True,True
6,6,26.250,0.125,True,True,True
7,7,12.625,0.125,True,True,True
8,8,19.125,0.125,True,True,True
9,9,61.000,0.125,True,True,True


,Condition,Seeds,Assembled,Censored,FinalConnected,MeanConnectedTimeFraction,MeanLateLargestGroupFraction,MeanLatePolarization,PostAssemblyBreakupRuns,MissedRepulsionPassesPerSecond,AssemblyFraction,MeanRestrictedConfirmationTime_s
0,Reference,20,20,0,20,0.806840,1.000,0.952283,0,0.014722,1.00,39.88125
1,Angular noise 0.01,20,19,1,19,0.764479,0.985,0.938354,1,0.009722,0.95,45.75625


In [5]:
gates = {
    "Long-horizon assembly demonstrated in reference cohort": bool(long_runs.AssemblySuccess.mean() >= .9),
    "Operational persistent-assembly definition supported": bool(definition["B_supported_in_tested_cohort"]),
    "Exact checkpoint restoration and source continuation": bool(checkpoints[verification_columns].all().all()),
    "Recording cadence independent of internal physics": bool(stride["AllChecksPassed"]),
    "Candidate timestep accepted across speeds 1–5": bool(resolution["CandidateAcceptedAcrossAllSpeeds"]),
}
verdict = "READY FOR LIGHT WITH DOCUMENTED LIMITATIONS" if all(gates.values()) else "NOT READY FOR LIGHT"
report = {
    "Verdict": verdict, "ScientificRuns": 30 + 45 + 40,
    "RecordingValidationRuns": 4, "CheckpointContinuationBranches": 90,
    "ReferenceRuns": 30, "ReferenceAssembled": int(long_runs.AssemblySuccess.sum()),
    "ReferenceCensored": int(long_runs.Censored.sum()),
    "AssemblyOnsetMedian_s": float(successful.AssemblyTime.median()),
    "AssemblyOnsetIQR_s": [float(successful.AssemblyTime.quantile(.25)), float(successful.AssemblyTime.quantile(.75))],
    "ReferencePostAssemblyBreakupRuns": int(successful.PostAssemblyBreakup.sum()),
    "Definition": "One geometric component continuously for at least 5 simulated seconds, monitored at every internal update",
    "ValidatedCheckpoints": len(checkpoints),
    "CandidateDt": .03125, "ComparisonDt": .015625,
    "CandidateAcceptedAcrossAllSpeeds": bool(resolution["CandidateAcceptedAcrossAllSpeeds"]),
    "ReferenceControl": {"speed": 1.0, "dt": .125, "fov_degrees": 360, "max_turn_rate": None, "angular_noise": 0},
    "NextStep": "Resolve temporal sensitivity and regenerate assembled checkpoints under the accepted internal protocol before adding scalar light-dependent speed.",
    "LightImplemented": False, "Gates": gates,
}
(DATA_DIR / "06_light_readiness_decision.json").write_text(json.dumps(report, indent=2) + "\n")
display(pd.DataFrame([{"Criterion": key, "Passed": value} for key, value in gates.items()]))
display(Markdown(f"""## Verdict: **{verdict}**

1. Random-start assembly is a meaningful part of the project: 30/30 reference seeds meet the persistent definition by 180 s, including 27 initially disconnected populations.
2. Median onset is 28.5 s, IQR 13.1875–55.125 s, range 0–86.75 s. Confirmation requires another 5 s. Seeds 0 and 2 were right-censored by the earlier 62.5-s horizon; they now assemble at 86.75 and 73.875 s.
3. There are zero censored reference seeds at 180 s. This is a finite-sample observation, not a universal assembly guarantee.
4. No reference run breaks apart after persistent assembly during 88.25–175 s of subsequent observation. The held-out noisy condition does include a later breakup, so stability should not be generalized to every variant.
5. dt=0.03125 does **not** satisfy the declared comparison criteria against 0.015625. MeanNND, group spread and, at speed 3, directional/connectivity measurements remain sensitive. Choosing the visually nicest timestep is not justified.
6. The future speed range is not numerically certified. At speed 5, missed passes fall from 0.0896/s to 0.0192/s; the remaining finest-step encounters are shallow. Zero exact sampled overlaps does not itself establish resolution, and the tested finest step is a reference rather than a proven limit.
7. Use definition B: one geometric component for at least 5 s, checked each internal update. A and B give the same onset/successes here; C admits 19+1 and advances onset in five seeds.
8. Thirty checkpoints restore state, parameters, IDs, absolute time and both RNGs exactly, reproduce 16 original continuation steps, and support identical paired 64-step controls.
9. Future light/control comparisons should begin from the **same socially assembled checkpoint**, while random initialization remains the official assembly experiment. Existing checkpoint validation applies to speed 1 and dt=0.125. Regenerate them under the final accepted simulation protocol.
10. Do not add light yet. Recording can already be separated from internal dynamics (e.g. dt=0.03125 and observations every four updates), but changing output frequency does not fix physics resolution. Retain the deterministic reference as a control; held-out noise does not justify adopting a new stochastic baseline.

**Next decision:** investigate the temporal social-response/update convention and further refinement, rather than tuning attraction or applying global cohesion. Then verify the accepted protocol across the intended speed range and rebuild paired checkpoints. This task stops here for review.
"""))


,Criterion,Passed
0,Long-horizon assembly demonstrated in referenc...,True
1,Operational persistent-assembly definition sup...,True
2,Exact checkpoint restoration and source contin...,True
3,Recording cadence independent of internal physics,True
4,Candidate timestep accepted across speeds 1–5,False


## Verdict: **NOT READY FOR LIGHT**

1. Random-start assembly is a meaningful part of the project: 30/30 reference seeds meet the persistent definition by 180 s, including 27 initially disconnected populations.
2. Median onset is 28.5 s, IQR 13.1875–55.125 s, range 0–86.75 s. Confirmation requires another 5 s. Seeds 0 and 2 were right-censored by the earlier 62.5-s horizon; they now assemble at 86.75 and 73.875 s.
3. There are zero censored reference seeds at 180 s. This is a finite-sample observation, not a universal assembly guarantee.
4. No reference run breaks apart after persistent assembly during 88.25–175 s of subsequent observation. The held-out noisy condition does include a later breakup, so stability should not be generalized to every variant.
5. dt=0.03125 does **not** satisfy the declared comparison criteria against 0.015625. MeanNND, group spread and, at speed 3, directional/connectivity measurements remain sensitive. Choosing the visually nicest timestep is not justified.
6. The future speed range is not numerically certified. At speed 5, missed passes fall from 0.0896/s to 0.0192/s; the remaining finest-step encounters are shallow. Zero exact sampled overlaps does not itself establish resolution, and the tested finest step is a reference rather than a proven limit.
7. Use definition B: one geometric component for at least 5 s, checked each internal update. A and B give the same onset/successes here; C admits 19+1 and advances onset in five seeds.
8. Thirty checkpoints restore state, parameters, IDs, absolute time and both RNGs exactly, reproduce 16 original continuation steps, and support identical paired 64-step controls.
9. Future light/control comparisons should begin from the **same socially assembled checkpoint**, while random initialization remains the official assembly experiment. Existing checkpoint validation applies to speed 1 and dt=0.125. Regenerate them under the final accepted simulation protocol.
10. Do not add light yet. Recording can already be separated from internal dynamics (e.g. dt=0.03125 and observations every four updates), but changing output frequency does not fix physics resolution. Retain the deterministic reference as a control; held-out noise does not justify adopting a new stochastic baseline.

**Next decision:** investigate the temporal social-response/update convention and further refinement, rather than tuning attraction or applying global cohesion. Then verify the accepted protocol across the intended speed range and rebuild paired checkpoints. This task stops here for review.


## Interpretation limits and reference roles

The readiness verdict refers to this implementation and the declared tests, not an impossibility of collective gradient sensing. Finite seeds and horizons, geometric connectivity, no subgroup lineage tracking, no empirical calibration, and the update-frequency convention limit inference. [Hein et al. (2015), Appendix 5.4](https://elifesciences.org/articles/10955) supports measuring formation time conceptually in a different model; it does not validate our numerical assembly times. [Couzin et al. (2002)](https://doi.org/10.1006/jtbi.2002.3065) and [Puckett et al. (2018)](https://doi.org/10.1038/s41598-018-26037-9) motivate the social framework and speed range. [Berdahl et al. (2013)](https://doi.org/10.1126/science.1225883) motivates the later scalar-light experiment, which remains unimplemented.
